# Working with GrIMP Velocity Products Using nisarVel and nisarVelSeries Classes
---

This notebook illustrates some of the capabilities of the `nisarVel` and `nisarVelSeries` classes for working with GrIMP velocity products. There are two classes, each derived from the same parent class so they have similar functionality. The main difference is that a `nisarVel` instance works with a velocity map for a single date. By constrast, the `nisarVelSeries` can incorporate any number of velocity products, so long as they have the same geometry (resolution and extent; e.g., all Greenand NSIDC-0725 velocity maps). For the terraSAR-X velocities, this means a `nisarVelSeries` instance can handle a single glacier box, but multiple `velSeries` instances would be needed to work with multiple glacier boxes. 

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('Features.md', encoding='utf-8').read()))

## Environment Setup

The following packages are needed to execute this notebook. The notebook has been tested with the `environment.yml` in the *binder* folder of this repository. Thus, for best results, create a new conda environment to run this and other other GrIMP notebooks from this repository. 

`conda env create -f binder/environment.yml`

`conda activate greenlandMapping`

`python -m ipykernel install --user --name=greenlandMapping`

`jupyter lab`

See [NSIDCLoginNotebook](https://github.com/fastice/GrIMPNotebooks/blob/master/NSIDCLoginNotebook.ipynb) for additional information.

The notebooks can be run on a temporary virtial instance (to start click [**binder**](https://mybinder.org/v2/gh/fastice/GrIMPNotebooks/HEAD?urlpath=lab)). See the github [README](https://github.com/fastice/GrIMPNotebooks#readme) for further details.

## Python Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import nisardev as nisar
import os
from urllib import request
import grimpfunc as grimp
import matplotlib.pyplot as plt
import dask
from dask.diagnostics import ProgressBar
ProgressBar().register()
import panel
import pyproj
import numpy as np
from datetime import datetime
import glob
panel.extension()

## Help

**Note to get help and see options for any of the GrIMP or other functions while the cursor is positioned inside a method's parentheses, click shift+Tab.**

## Trouble Shooting

NSIDC limits the number of simultaneous connections. As a result, a download can sometimes fail, especially if multiple notebooks are downloading or the ```num_workers``` is set to large. In these cases, try rerunning with only a single notebook downloading or ```num_workers=2``` (current default).

## Local Access

This notebook is designed to pull the data directly from NISDC. If the data have already been downloaded, the notebook can be modified to run with local file paths by replacing the http links in ```myCogs``` (see below) with file paths formatted as shown below (use a wildcard '\*' for the band remove the '.tif' extension) and setting the ```url``` keyword in the read commands to ```False```.

## Locate Data

Velocity GrIMP velocity products are as stored at NSIDC in cloud-optimized geotif (COG) format with each compenent stored as separate band (e.g., vx, vy). In this notebook, we focus on on the velocity data, but the error and compenents can be similarly processed.

For reading the data, the products are specified with a single root file name (e.g., for *filename.vx(vy).othertext.tif*). For example, the version 3 annual mosaic December 2017 to November 2017 is specified as `GL_vel_mosaic_Annual_01Dec17_30Nov18_*_v03.0`. For locally stored files, the corresponding path to the data must be provided. For remote data, the https link is required. 

## Login to EarthData/NSIDC

Unless the data have already been downloaded, users will need to sign in to NSIDC/EarthData to run the rest of the notebook. For locally accessed data that have already been downloaded, set `skipLogin=True` in the cell below. 

In [ ]:
env = dict(GDAL_HTTP_COOKIEFILE = os.path.expanduser('~/.grimp_download_cookiejar.txt'),
            GDAL_HTTP_COOKIEJAR = os.path.expanduser('~/.grimp_download_cookiejar.txt'))
os.environ.update(env)
myLogin = grimp.NASALogin()  # If login appears not to work, try rerunning this cell
display(myLogin.view())

This cell will pop up a search tool for the gimp products, which will run a predefined search for the annual products. While in principle, other products (e.g., six-day to quarterly can be retrieved, the rest of the notebook will need some modifications to accomodate).

In [ ]:
# For some environments the tool is unresponsive (i.e., search button doesn't work) - this can often be fixed by re-running this cell
myUrls = grimp.cmrUrls(mode='nisar')  # Subsetter mode is required for subsetting.
myUrls.initialSearch()

Since there are multiple bands per velocity product, the read routines expect names of the form `https://data.nsidc.earthdatacloud.nasa.gov/nsidc-cumulus-prod-protected/MEASURES/NSIDC-0725/5/2014/12/01/GL_vel_mosaic_Annual_01Dec14_30Nov15_*_v05.0`. The names are conditioned this way as shown below. 

In [ ]:
myCogs = myUrls.getCogs(replace='vv', removeTiff=True)

## Point Data

In several of its examples, this notebook uses a recently collected set of GPS points ([Hvidberg et al., 2020](https://tc.copernicus.org/articles/14/3487/2020/)) from the Northeast Greenland Ice Stream (NEGIS). The following cell will read these data from a subdirectory included with this repository.

In [ ]:
lltoxy = pyproj.Transformer.from_crs(4326,3413)
with open('GPSpoints/NEGIS-GPS.txt') as fp:
    lines = []
    for line in fp:
        lines.append([float(x) for x in line.split()[0:5]])
    latGPS, lonGPS, zGPS, vxGPS, vyGPS = np.array(lines).transpose()
    xGPS, yGPS = lltoxy.transform(latGPS, lonGPS)

In the examples below, a bounding box for these points will be used to crop the data. The box is calculated as:

In [ ]:
pad = 20e3 # pad box by 20 km in each dimension and round to nearest km
values = np.around([np.min(xGPS) - pad, np.min(yGPS) - pad,
                    np.max(xGPS) + pad, np.max(yGPS) + pad], -3)
xyBounds = dict(zip(['minx', 'miny', 'maxx', 'maxy'], values))
xyBounds

## To Chunk or Not

There are two ways to read the data, which are controlled by the parameter `useStack`:
1) `useStack=True` (default): In this case, each band of the subsetted data is loaded as a single file read operation (e.g., no chunks in xy). For most cases, this is the faster option because there is no dask overhead and the single reads tend to be faster. This is the preferred option for anything that involves reading in data with `loadRemote` and doing repeated operations on the result. **Note: with this option, the keyword `chunks` will be ignored.**
2) `useStack=False`: In this case, the data are chunked with rio-xarray and dask, which can add 10s of seconds or more overhead to set up the xarray, which slows the lazy reads. There are only a few use cases in which this mode would be preferred. For example, examining only a few points in a large subset **without** using `loadRemote`, so that as the data are input on the fly, only the chunks surrounding the points would be read. In most cases, it's best to minimize the subset area (e.g., for one glacier) or use multiple subsets for widely spaced glaciers.

In [ ]:
useStack=True

## Number of Workers

With Dask you can use `numWorkers` to specify multiple parallel threads, which can speed up downloads. It can also cause the download to fail (with something that looks like a file not found error) if the server decides it's receiving too many concurrent requests. The criteria under which this happens are unclear, and whether it has to do with the number of connections, open files, etc. This means the results could differ for the type of data product/access. For example, downloading a large number of TSX files (many file open operations per unit time) might cause things to break before the case where large pieces are pulled from full ice sheet mosaics (files are held open for long periods). Beyond `numWorkers=8`, the point of diminishing returns is approached. In general, `numWorkers=4` will be fairly robust, and provide good performance. But you can experiment by setting `numWorkers` below.  Note: this discussion is most applicable to network reads. For local file systems, the speedup may be substantially less due to file contention.

**If a download fails, try re-running. If it still fails, try reducing value of `numWorkers`**

In [ ]:
numWorkers = 4

## NISARVel

The  nisar class is used to read, display, velocity maps with a single time stamp. In this example, the `readSpeed=False` (default) forces the speed to be calculated from the individual components rather than read from a file, which is much quicker.

In [ ]:
myVel = nisar.nisarVel(numWorkers=numWorkers)
myVel.readDataFromTiff(myCogs[3], url=True, readSpeed=False, useStack=useStack)

The data in the above map are stored as an Xarray, `velMap.subset`. In this case, the subset is the full map of Greenland, which is > 1GB and could take a while to download (there more than 300 similars maps a 6-day resolution). Because of the lazy open mentioned above, the data have not been downloaded or read from disk yet. 

As a result, the data can be subset at this point to cover just the area spanned by the tiepoints by:

In [ ]:
myVel.subsetVel(xyBounds)

The cloud-optimized geotiffs using 512x512 pixel tiling scheme. Although the area-of-interest in the above data map is smaller than this, for each band in this case, data have to be pulled from 3 tiles, which is still far more efficient that reading the full 7585 pixel with in the full Greenland map above.

The speed overplotted by the GPS points loaded above can displayed as with units of either m or k for the x and y axes. In addition the title date can be toggled between the middle date and the range of dates by setting the boolean `midDate` keyword as shown here:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
scale = {'m': 1., 'km': 0.001}
for ax, units, midDate in zip(axes, ['m', 'km'], [True, False]):
    myVel.displayVel(ax=ax, units=units, midDate=midDate)
    ax.plot(xGPS * scale[units], yGPS * scale[units], 'ro')
fig.suptitle('Speed with Different Date Formats (central vs first&last) and xy units (m, km) ', fontsize=15)
fig.tight_layout()

At this point, the lazy open still applies. In making the above plot, the data were loaded for the plot automatically by Dask. Depending on cache sizes, the data could be flushed and have to be re-downloaded before the next operation. If the data can comfortably fit in memory, it can be better to download them. As noted above, in addition to the Xarray, the data are broken out as individual numpy arrays (eg., `mYVel.vx, myVel.vy`. At this point, however, they are Dask arrays, meaning they haven't actually been executed (except on demand as in the above example).

In [ ]:
myVel.subsetVel(xyBounds) # reset subset to avoid problem with out of order execution
print(type(myVel.vx))
print(f'vx[100, 100] type: {type(myVel.vx)}')

Instead of the velocity, we get the dask array, which contains instructions on how to retrieve the data when needed. In this case, we can get the value by forcing Dask to compute it by:

In [ ]:
print(myVel.vx[100,100].compute())

If we force the data into memory, the component variables become regular numpy arrays and the subset becomes a regular xarray.

In [ ]:
myVel.loadRemote()
print(f'vx[100, 100] type: {type(myVel.vx)}')
myVel.vx[100,100]

In this example, with a fast internet connection there maybe no real advantage to downloading the data. But for larger data sets that take 10s of seconds to minutes to download, it can speed things up dramatically. It also allows the user to avoid complexities of dask (for example if they just want the value at a point). For many operations, however, the dask operations are transparent to the user (for example in plotting the data as shown above). 

## Interpolation

The velocity data can be interpolated as:

In [ ]:
vxInterp, vyInterp, vvInterp = myVel.interp(xGPS, yGPS, units='m')

If we wanted to avoid the converstion from lat/lon to x, y coordinates above, we could simply passed the EPSG code for the coordinates as:

In [ ]:
vxInterp1, vyInterp1, vvInterp1 = myVel.interp(latGPS, lonGPS, sourceEPSG=4326)

And to have the result returned as an Xarray instead of numpy, the call would be : 

In [ ]:
velInterp = myVel.interp(latGPS, lonGPS, sourceEPSG=4326, returnXR=True)

As expected, the results are all identical.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, speed, title in zip(axes, [vvInterp, vvInterp1, np.squeeze(velInterp.sel(band='vv'))],
                       ['x,y Interp', 'lat/lon Interp', 'Xarray']):
    ax.plot(speed, '-o')
    ax.set_title(title)
    ax.set_xlabel('Point Number')
    ax.set_ylabel('Speed (m/yr)')
fig.tight_layout()

## Velocity Series

In the above examples, only a single velocity product was used. Using `nisarVelSeries` class in place of the `nisarVel` several products can be read, subsetted, and downloaded. From this cell to the end of the notebook, the results are the same irrective of whether `loadRemote` is called. With `loadRemote`, however, the steps run about twice as fast using a local copy of the data. A `nisarVelSeries` can be instantiated and loaded with the following steps, which also include subsetting and data loading. 

In [ ]:
start = datetime.now()
myVelSeries = nisar.nisarVelSeries(numWorkers=numWorkers)
myVelSeries.readSeriesFromTiff(myCogs, url=True, readSpeed=False, useStack=useStack)
#
myVelSeries.subsetVel(xyBounds) # Apply subset
#
myVelSeries.loadRemote()

The velocity series work the same way as the single velocity products, except in some cases a date needs to be specified to select a layer and the method `displayVelForDate` is used (if no date given it defaults to the first date). For example, the first 6 annual products can be displayed by:

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20,10))
for date, ax in zip(myVelSeries.time[0:6], axes.flatten()):
    myVelSeries.displayVelForDate(date=date, band='vv', ax=ax, units='km')
fig.suptitle('Time Series of Speed', fontsize=15)
fig.tight_layout()

## Plot Points and Profiles

A ```VelSeries``` object can directly plot the time series for a point or a profile for a single time as demonstrated in the next cell.

In [ ]:
# Generate a horizontal profile (fixed y) in units of km
xprof = np.arange(200, 280, .25)
yprof = np.full(xprof.shape, -1530)
xpt, ypt = 250, -1530
#
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
#
# Display velocity and locations of point and profile
myVelSeries.displayVelForDate('2017-06-01', ax=axes[0], units='km', colorBarPosition='top', title='')
axes[0].plot(xprof, yprof, color='w')
axes[0].plot(xpt, ypt, '*r', markersize=12)
#
# plot time series for point
myVelSeries.plotPoint(xpt, ypt, 'r-*', ax=axes[1], markersize=12, units='km')
myVelSeries.labelPointPlot(ax=axes[1], title='Speed Near the Center of the Ice Stream')
# plot profile for each date
for date in myVelSeries.time:
    myVelSeries.plotProfile(xprof, yprof, ax=axes[2], units='km', date=date)
# label with all fontsizes increased by 30%
myVelSeries.labelProfilePlot(ax=axes[2], title='Profiles of Speed Acrossthe Ice Stream', fontScale=1.3)
axes[2].legend()
fig.tight_layout()

In this example, the speeds are only changing slowly as indicated by the time series and sequence of profiles that overplot each other.

## Operations on Velocity Series

Various operations can be applied in time and space to the velocity series. For example, at each point, the mean and standard devation along with the number of valid points for velocity time series are computed as:

In [ ]:
%%time
#Compute Stats
velMean = myVelSeries.mean()
velSigma = myVelSeries.stdev()
velCount = myVelSeries.numberValid()

In the following example, the mean, standard deviation, and number of valid for each location are calculated. Note for the annual data in this fairly benign region, all points are valid (n=7). Each result is returned as a velocity series with only 1 time. The results can be plotted by:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,4.5))
velMean.displayVelForDate(ax=axes[0], units=units, midDate=True, colorBarLabel='Mean Speed (m/yr)')
velSigma.displayVelForDate(ax=axes[1], date=None, units=units, midDate=True, vmin=0, vmax=3, autoScale=False, colorBarLabel='Sigma Speed (m/yr)')
velCount.displayVelForDate(ax=axes[2], date=None, units=units, midDate=True, vmin=0, vmax=7, autoScale=False, colorBarLabel='Number Valid', extend='neither')
fig.suptitle('Mean, Sigma, and Number of Valid Points', fontsize=16)
fig.tight_layout()

The anomalies (annual value - mean) can be calcuated for each time slices in the series as:

In [ ]:
velAnomaly = myVelSeries.anomaly()
# Plot the anomaly for each year
fig, axes = plt.subplots(2, 3, figsize=(20,10))
for date, ax in zip(velAnomaly.time, axes.flatten()):
    velAnomaly.displayVelForDate(date, ax=ax, units='km', vmin=-2, vmax=2, autoScale=False, colorBarLabel='Speed Anomaly (m/yr)', extend='both')
fig.suptitle('Speed Anomalies by Year', fontsize=16)
fig.tight_layout()

The spatial mean for each time in the series can be computed as: 

In [ ]:
meanXR = myVelSeries.meanXY(returnXR=True) # returnXR=False will return x, y, speed means as nparrays
#
fig, ax = plt.subplots(1, 1, figsize=(10,7))
for band in meanXR.band:
    ax.plot(meanXR.time, meanXR.sel(band=band), '-o', label=band.item())
    ax.set_title('Mean Velocity by Year')
ax.legend();

The standard deviation for the anomalies computed above is evaluated and plotted by year using: 

In [ ]:
stdevAnomaly = velAnomaly.stdevXY(returnXR=True) # returnXR=False will return x, y, speed means as nparrays
#
fig, ax = plt.subplots(1, 1, figsize=(10,7))
for band in stdevAnomaly.band:
    ax.plot(stdevAnomaly.time, stdevAnomaly.sel(band=band), '-o', label=band.item())
    ax.set_title('Standard Deviation of Velocity Anomaly')
ax.legend();

This plot shows a) the precsion of the data and b) any true velocity change.

# Interpolation

In the examples above, the interpolation was for a single time.

In [ ]:
vxm, vym, vvm = myVelSeries.interp(xGPS, yGPS, date=None, units='m') # x,y coordinates are passed in as m
#
fig, axes = plt.subplots(1, 3, figsize=(15,5))
for ax, band, vInterp in zip(axes, ['vx', 'vy', 'vv'], [vxm, vym, vvm]):
    for date, v in zip(myVelSeries.time, vInterp):
        ax.plot(v, '.', label=date.strftime('%Y'))
    ax.legend()
    ax.set_title(f'Interpolated {band}')
    ax.set_xlabel('Point Number')
    ax.set_ylabel(f'{band} m/yr', fontsize=13)

The example repeats the above example except that it a) interpolates directly from the GPS lat/lon values (```sourceEPSG=4326```) and b) the results are returned as a single xarray (```returnXR=True```) rather than multiple numpy arrays.  

In [ ]:
vPts = myVelSeries.interp(latGPS, lonGPS, date=None, units='m', returnXR=True, sourceEPSG=4326)
fig, axes = plt.subplots(1, 3, figsize=(15,5))
for ax, band in zip(axes, vPts.band):
    for t in vPts.time:
        year = str(np.datetime64(t.item(0), 'ns'))[0:4]
        ax.plot(vPts.sel(band=band, time=t), '.', label=year)
    ax.legend()
    ax.set_title(f'Interpolated {band.data}')
    ax.set_xlabel('Point Number')
    ax.set_ylabel(f'{band.data} m/yr', fontsize=13)

## Subsetting the data multiple times

When the data are loaded into a series, the details of the full map are stored even when the data are subsetted. As a result, its always possible to create a new subset without creating a new series (the prior subset will be lost) as shown by this example, which plots the last 6 layers and demonstrates the `axisOff` keyword to turn of the *xy* axes.

In [ ]:
# Shift box to faster moving parts of the ice stream
newBounds = {'minx': 391000.0, 'maxx': 483000.0, 'miny': -1124000.0, 'maxy': -1053000.0}
#
myVelSeries.subsetVel(newBounds) # Apply subset
myVelSeries.loadRemote()
#
fig, axes = plt.subplots(2, 3, figsize=(20,10))
for date, ax in zip(myVelSeries.time[1:8], axes.flatten()):
    myVelSeries.displayVelForDate(date=date, band='vv', scale='log', ax=ax, units='km', axisOff=True)
fig.suptitle('Time Series of Speed', fontsize=15)
fig.tight_layout()

## Inspect the Data

An interactive plot to inspect the data can be generated as **(if subsequent cells have re-subsetted the data, this step may need to be rerun to work)**:

In [ ]:
# Reapply bounds if needed to avoid out of order changing of subsets
if myVelSeries.boundingBox() != newBounds:
    myVelSeries.subsetVel(newBounds) # Apply subset
    myVelSeries.loadRemote()
myVelSeries.inspect(imgOpts={'clim': (0,2000), 'logz': True, 'cmap': 'hsv'})

## Saving the Data

Some downloads could take several minutes (e.g., all 6/12 day maps), in which case it useful to be able to save the data to a single netCDF file for later use. The downloaded subset can be saved in a netcdf and reloaded for to `velSeries` instance for later analysis. Note if the data have been subsetted, **ONLY** the subset will be saved (~11MB in this example). If not, the entire Greeland data set will be saved (370GB). Before saving, the bounds will be set to the original values, forcing a new download to overwrite the previous.

In [ ]:
myVelSeries.subsetVel(xyBounds) # Apply the original subset
myVelSeries.loadRemote()
myVelSeries.toNetCDF('xyBounds.nc')
# Now reload the data
myVelReload = nisar.nisarVelSeries()
myVelReload.readSeriesFromNetCDF('xyBounds.nc')
myVelReload.loadRemote()
os.remove('xyBounds.nc')  # Cleanup and remove file, comment out this line to keep file

The next cell reloads and plots the data.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20,10))
for date, ax in zip(myVelReload.time, axes.flatten()):
    myVelReload.displayVelForDate(data=date, band='vv', ax=ax, units='km')
fig.suptitle('Time Series of Speed Reloaded from NetCDF', fontsize=15)
fig.tight_layout()

The next cell plots the reloaded velocity map and also demonstrates the log color table:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
myVelReload.displayVelForDate('2017-06-01', ax=axes[0], units=units, midDate=True)
myVelReload.displayVelForDate('2017-06-01', ax=axes[1], units=units, midDate=False, vmin=1, vmax=1000, scale='log', percentile=99)
fig.tight_layout()